# Real-Time Multi-Frame VSR Training
This notebook covers downloading the dataset via Kaggle, preprocessing it on Colab, training the network, and exporting the ONNX model.

### 1. Setup & Mount Google Drive
We mount Google Drive to save our trained model checkpoints so they aren't lost if Colab disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/VSR_Checkpoints

### 2. Clone Repository & Install Requirements

In [ ]:
!git clone https://github.com/DarshanSamuel/VSR_RealTime.git
%cd VSR_RealTime
!pip install -r requirements.txt

### 3. Kaggle Setup & Dataset Download
This cell configures your Kaggle Token and downloads the dataset automatically.

In [ ]:
import os

# Set the Token exactly as Kaggle instructed
token = "KAGGLE_API_TOKEN"
os.environ['KAGGLE_API_TOKEN'] = token

!mkdir -p ~/.kaggle
!echo {token} > ~/.kaggle/access_token
!chmod 600 ~/.kaggle/access_token

# REPLACE THIS with the actual Kaggle slug for the 9.26GB dataset you found
kaggle_slug = "username/dataset-name"

!kaggle datasets download -d {kaggle_slug}

!echo "Unzipping dataset..."
!unzip -q *.zip -d /content/vimeo_septuplet
!echo "Done!"

### 4. Preprocess Dataset

In [ ]:
!PYTHONPATH=. python src/vsr/dataset.py --vimeo /content/vimeo_septuplet --out /content/processed_dataset --split train --frames 7

### 5. Stage A: Single-Frame Pretraining

In [ ]:
!PYTHONPATH=. python src/vsr/train.py --data /content/processed_dataset --out /content/drive/MyDrive/VSR_Checkpoints --stage A

### 6. Stage B: Recurrent Finetuning

In [ ]:
!PYTHONPATH=. python src/vsr/train.py --data /content/processed_dataset --out /content/drive/MyDrive/VSR_Checkpoints --stage B

### 7. Export to ONNX

In [ ]:
!pip install onnxscript
!PYTHONPATH=. python src/vsr/export_onnx.py --checkpoint /content/drive/MyDrive/VSR_Checkpoints/model_stage_B_epoch_100.pth --out /content/drive/MyDrive/VSR_Checkpoints/vsr_model.onnx

print("Training and export complete! You can now download vsr_model.onnx from your Google Drive.")